# Experimental Appendix — Benchmark Details

Companion document to the paper. Covers the **Physics-Informed Neural Network (PINN)** benchmark problem in full detail (PDE, network architecture, loss formulation, validation), the other model classes, and the hardware/software stack used.

All code lives in [`benchmarks/deep_objectives.py`](../benchmarks/deep_objectives.py) (single-fidelity objectives) and [`benchmarks/multifidelity_objectives.py`](../benchmarks/multifidelity_objectives.py) (fidelity-aware wrappers).

---

## 1. The PINN Benchmark — 1D Heat Equation

We use a **physics-informed neural network** for the 1D heat-equation initial-boundary-value problem (IBVP). The Heat equation is the simplest nonlinear-free parabolic PDE with a well-known analytical solution, which lets us measure the PINN's accuracy against ground truth.

### 1.1 PDE and boundary conditions

Find $u(x, t) : [0, 1] \times [0, T_{\max}] \to \mathbb{R}$ such that:

$$
\frac{\partial u}{\partial t} \;=\; \kappa \, \frac{\partial^2 u}{\partial x^2}
\qquad \text{on } x \in (0,1), \; t \in (0, T_{\max}]
$$

Initial condition (IC):
$$
u(x, 0) = \sin(\pi x) \qquad \text{for } x \in [0, 1]
$$

Dirichlet boundary conditions (BC):
$$
u(0, t) = u(1, t) = 0 \qquad \text{for } t \in [0, T_{\max}]
$$

with thermal diffusivity $\kappa = 0.1$ and time horizon $T_{\max} = 1.0$ (both fixed across the benchmark).

### 1.2 Analytical solution (used as ground truth)

By separation of variables, the closed-form solution is:
$$
u_{\text{true}}(x, t) = e^{-\kappa \pi^2 t} \cdot \sin(\pi x)
$$

This decays exponentially from $\sin(\pi x)$ at $t = 0$ toward zero at $t = T_{\max}$. The factor $e^{-\kappa \pi^2 t}$ is the time-evolution of the first Fourier mode of the IC, and the spatial profile remains a half-sine throughout.

### 1.3 Why this problem?

Three reasons it's a useful HPO benchmark:

1. **Reproducibility.** The analytical solution provides a *clean*, *deterministic* validation target: the PINN's mean-squared-error against $u_{\text{true}}$ on a fixed grid is the loss minimized by HPO. No statistical noise from train/test splits.
2. **Standard PINN testbed.** The 1D heat equation is the canonical "smoke test" problem in the PINN literature (Raissi et al. 2019, Karniadakis 2017). Results here translate to harder problems.
3. **Non-trivial loss landscape.** Even though the solution is smooth, the *PINN training objective* (a weighted sum of PDE-residual, IC, and BC losses) is non-convex over the hyperparameters (architecture, learning rate, loss weighting). This makes it a genuine HPO task.

### 1.4 Network architecture — `_MLPSinusoidal`

A standard MLP that takes the space-time coordinate $(x, t) \in \mathbb{R}^2$ as input and outputs $u(x, t) \in \mathbb{R}$.

```
Input  (x, t)  ∈ ℝ²
  │
  ├─ Linear(2 → hidden_dim) → activation
  │
  ├─ Linear(hidden_dim → hidden_dim) → activation     ┐  repeat
  │  ...                                              ├─  (num_layers − 1) times
  ├─ Linear(hidden_dim → hidden_dim) → activation     ┘
  │
  └─ Linear(hidden_dim → 1)
Output  û(x, t) ∈ ℝ
```

`hidden_dim` and `num_layers` are HPO parameters (Section 4 below). `activation` is one of `tanh`, `gelu`, `elu`. The name `MLPSinusoidal` is historical (an early version used sinusoidal features); the current implementation is a standard fully-connected MLP.

### 1.5 Loss function — data + physics decomposition

The standard PINN training loss in the literature follows a **data + physics** decomposition:

$$
\mathcal{L}_{\text{train}} \;=\; w_{\text{data}} \cdot \mathcal{L}_{\text{data}} \;+\; w_{\text{phys}} \cdot \mathcal{L}_{\text{physics}}
$$

where $\mathcal{L}_{\text{data}}$ is the MSE between the network's output and any **observed values** of $u$ (measurements, initial values, boundary values), and $\mathcal{L}_{\text{physics}}$ is the MSE of the **PDE residual** at sampled interior points.

**Forward vs inverse problem distinction.** For the 1D heat equation we solve a *forward* problem: given the PDE, the IC, and the BC, predict $u(x,t)$ everywhere in the interior. There are no scattered observations inside the domain, so the "data" term reduces to just the IC and BC. Our concrete loss is therefore:

$$
\boxed{
\mathcal{L}_{\text{train}}
\;=\;
\underbrace{\mathcal{L}_{\text{IC}} + w_{\text{bc}} \cdot \mathcal{L}_{\text{BC}}}_{\mathcal{L}_{\text{data}}\,\text{(known boundary measurements)}}
\;+\;
\underbrace{w_{\text{phys}} \cdot \mathcal{L}_{\text{phys}}}_{\mathcal{L}_{\text{physics}}\,\text{(PDE residual)}}
}
$$

For an *inverse* PINN problem (e.g., a glacier ice-flow model with scattered satellite-derived thickness measurements), an additional interior data term

$$
\mathcal{L}_{\text{interior-data}} = \frac{1}{N} \sum_{i=1}^N \big( \hat u_\theta(x_i, t_i) - u_{\text{obs},i} \big)^2
$$

would also appear, and $w_{\text{data}}$ would weight the relative trust in noisy observations vs the PDE. Our forward heat-equation problem has no such interior observations by design, since the analytical solution lets us evaluate HPO methods *deterministically* against ground truth (Section 1.6). The three components of our training loss — IC, BC, and PDE residual — each compute on a different sample of points and are described below.

**Physics-residual loss** (PDE compliance, interior):

$$
\mathcal{L}_{\text{phys}}(\theta) = \frac{1}{N_{\text{coll}}} \sum_{i=1}^{N_{\text{coll}}} \big| \, \partial_t \hat u_\theta(x_i, t_i) - \kappa \, \partial_{xx}^2 \hat u_\theta(x_i, t_i) \, \big|^2
$$

evaluated at $N_{\text{coll}}$ random collocation points $(x_i, t_i) \sim \mathcal{U}([0,1] \times [0, T_{\max}])$. The derivatives $\partial_t$ and $\partial_{xx}$ are computed via PyTorch **automatic differentiation** through the network — this is the defining trick of PINNs.

**Initial-condition loss**:

$$
\mathcal{L}_{\text{IC}}(\theta) = \frac{1}{N_{\text{ic}}} \sum_{j=1}^{N_{\text{ic}}} \big( \hat u_\theta(x_j, 0) - \sin(\pi x_j) \big)^2
$$

evaluated at $N_{\text{ic}} = 80$ uniformly-spaced points on $x \in [0, 1]$ at $t = 0$.

**Boundary-condition loss**:

$$
\mathcal{L}_{\text{BC}}(\theta) = \frac{1}{N_{\text{bc}}} \sum_{k=1}^{N_{\text{bc}}} \big[\, \hat u_\theta(0, t_k)^2 + \hat u_\theta(1, t_k)^2 \,\big]
$$

evaluated at $N_{\text{bc}} = 60$ uniformly-spaced points on $t \in [0, T_{\max}]$.

**Total training loss**:

$$
\mathcal{L}_{\text{train}}(\theta) = w_{\text{phys}} \cdot \mathcal{L}_{\text{phys}}(\theta) \;+\; \mathcal{L}_{\text{IC}}(\theta) \;+\; w_{\text{bc}} \cdot \mathcal{L}_{\text{BC}}(\theta)
$$

The weights $w_{\text{phys}}$ and $w_{\text{bc}}$ are HPO parameters — their tuning is exactly the kind of problem HPO methods should solve. The IC weight is fixed at 1.0 as the reference.

### 1.6 Validation metric — MSE vs analytical solution

After training, the PINN is evaluated on a **fixed** $40 \times 20 = 800$-point grid:

$$
\mathcal{X}_{\text{val}} = \{0, \tfrac{1}{39}, \tfrac{2}{39}, \dots, 1\} \times \{0, \tfrac{T_{\max}}{19}, \tfrac{2 T_{\max}}{19}, \dots, T_{\max}\}
$$

The HPO objective minimized by the tuner is:

$$
\mathcal{L}_{\text{HPO}}(\theta) = \min_{e \in [1, E]} \;\; \text{MSE}\big( \hat u_\theta^{(e)}, u_{\text{true}} \big) \quad \text{over the validation grid}
$$

where $\hat u_\theta^{(e)}$ is the network at training epoch $e$, and $E$ is the maximum epochs (an HPO parameter). Best-so-far validation MSE is logged every $E/20$ epochs with early-stopping patience of 50 such checks.

### 1.7 Hyperparameter search space (11 dimensions)

| Dimension          | Type         | Range / choices              | Role |
|--------------------|--------------|------------------------------|------|
| `lr`               | Float (log)  | $[10^{-5}, 10^{-1}]$         | Adam/AdamW/RMSprop learning rate |
| `weight_decay`     | Float (log)  | $[10^{-7}, 10^{-3}]$         | L2 regularization strength |
| `hidden_dim`       | Int          | $[16, 128]$                  | MLP hidden width |
| `num_layers`       | Int          | $[2, 6]$                     | MLP depth |
| `activation`       | Categorical  | {tanh, gelu, elu}            | nonlinearity between layers |
| `optimizer`        | Categorical  | {adam, adamw, rmsprop}       | training optimizer |
| `scheduler`        | Categorical  | {cosine, step, none}         | LR schedule |
| `physics_weight`   | Float (log)  | $[10^{-2}, 10^{2}]$          | $w_{\text{phys}}$ in total loss |
| `bc_weight`        | Float (log)  | $[10^{-1}, 10^{1}]$          | $w_{\text{bc}}$ in total loss |
| `n_collocation`    | Int          | $[500, 2500]$                | $N_{\text{coll}}$ interior points |
| `epochs`           | Int          | $[100, 500]$                 | maximum training epochs |

This is **mixed-type** (5 floats, 4 of which log-scaled; 3 ints; 3 categoricals) and **medium-dimensional** — exactly the regime where HDGPSO is theoretically expected to be competitive.

### 1.8 Multi-fidelity formulation (for HDGPSO-MF)

The fidelity parameter $f \in [0, 1]$ scales two dimensions of the training cost:

$$
N_{\text{coll}}(f) = \max\big(100, \; \lfloor f \cdot \text{n\_collocation} \rfloor \big)
$$
$$
E(f) = \max\big(20, \; \lfloor f \cdot \text{epochs} \rfloor \big)
$$

A probe at $f = 0.3$ costs roughly $0.3^2 = 0.09 \times$ the full evaluation (collocation samples scale linearly, epochs scale linearly with each epoch's cost). For budget accounting we use the linear convention $\text{cost}(f) = f$ — i.e., a probe at $f = 0.3$ consumes 0.3 budget units even though it's substantially cheaper than 0.3 of a full eval. This is a conservative cost model that under-counts the real savings, making HDGPSO-MF's reported budget efficiency a lower bound.

### 1.9 Other PINN implementation details

- **Framework**: PyTorch 2.10 + CUDA 12.8.
- **Initialization**: PyTorch default (uniform Xavier for `nn.Linear`).
- **Determinism**: `torch.manual_seed(seed)` is set at the start of each evaluation. CUDA non-determinism (cuDNN convolution algorithm selection, atomic operations) can still introduce small per-seed variance, which is why we use 3 seeds per cell.
- **Early stopping**: validation MSE checked at most $\lceil E / \max(E/20, 1) \rceil = 20$ times during training; stop after 50 consecutive checks with no improvement.
- **Loss safety**: if `loss.backward()` produces NaN/Inf (e.g., from too-aggressive learning rate), the cell returns `inf`, contributing rank 8 in that (dataset, seed) row.

---

## 2. Non-PINN Benchmark Tasks

The PINN problem is one of nine valid `(dataset, model)` pairs in the benchmark. The others are standard sklearn tasks.

### 2.1 Datasets and attributions

All three real datasets are loaded through scikit-learn's `datasets` module and originate from the [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml) (Dua & Graff, 2019). The synthetic PINN-Heat task is generated programmatically (Section 1).

| Dataset           | Task type      | Samples | Features | sklearn loader | Original source |
|-------------------|----------------|---------|----------|---------------|------------------|
| `breast_cancer`   | Classification (binary) | 569  | 30  | `load_breast_cancer` | Street, Wolberg & Mangasarian (1993) — *Wisconsin Diagnostic Breast Cancer (WDBC)*, IS&T/SPIE Proc. vol. 1905. |
| `wine`            | Classification (3-class)| 178  | 13  | `load_wine` | Aeberhard, Coomans & de Vel (1992), James Cook Univ. Tech. Rep. 92-02. Original chemical analysis by M. Forina et al., Univ. of Genoa. |
| `diabetes`        | Regression              | 442  | 10  | `load_diabetes` | Efron, Hastie, Johnstone & Tibshirani (2004) — "Least Angle Regression," *Ann. Statistics* 32(2). |
| `pinn_heat`       | PDE (synthetic)         | n/a  | n/a | (custom) | Standard 1D heat-equation IBVP; see §1 of this appendix. |

**Acknowledgments to dataset contributors.** We gratefully acknowledge the UCI ML Repository and the original dataset contributors. The Wisconsin Diagnostic Breast Cancer dataset is courtesy of Dr. William H. Wolberg, the University of Wisconsin Hospitals, and Olvi L. Mangasarian. The wine recognition data is courtesy of M. Forina (Università degli Studi di Genova). The diabetes dataset is from the supplementary materials of Efron et al. (2004), originally collected for studying diabetes progression.

**Licensing.** All three datasets are publicly available under permissive terms via UCI / scikit-learn. They are widely used as canonical benchmarks in the ML literature and impose no usage restrictions for academic comparison.

### 2.2 Model classes and search spaces

**RandomForest** (5 hyperparameters, discrete-heavy):

| Dim                 | Type        | Range              |
|---------------------|-------------|--------------------|
| `n_estimators`      | Int         | [20, 300]          |
| `max_depth`         | Int         | [2, 20]            |
| `min_samples_split` | Int         | [2, 20]            |
| `min_samples_leaf`  | Int         | [1, 20]            |
| `max_features`      | Categorical | {sqrt, log2, 0.5, 1.0} |

**GradientBoosting** (5 hyperparameters, mixed):

| Dim                 | Type        | Range              |
|---------------------|-------------|--------------------|
| `n_estimators`      | Int         | [20, 300]          |
| `learning_rate`     | Float (log) | $[10^{-3}, 0.3]$   |
| `max_depth`         | Int         | [2, 10]            |
| `min_samples_split` | Int         | [2, 20]            |
| `subsample`         | Float       | [0.5, 1.0]         |

**MLP** (12 hyperparameters, high-dim mixed):

| Dim            | Type        | Range / choices                         |
|----------------|-------------|----------------------------------------|
| `lr`           | Float (log) | $[10^{-5}, 10^{-1}]$                   |
| `weight_decay` | Float (log) | $[10^{-6}, 10^{-2}]$                   |
| `batch_size`   | Int (log)   | [16, 256]                              |
| `hidden_1`     | Int         | [32, 512]                              |
| `hidden_2`     | Int         | [16, 256]                              |
| `dropout_1`    | Float       | [0.0, 0.6]                             |
| `dropout_2`    | Float       | [0.0, 0.6]                             |
| `activation`   | Categorical | {relu, gelu, tanh, elu}                |
| `optimizer`    | Categorical | {adam, adamw, sgd, rmsprop}            |
| `scheduler`    | Categorical | {cosine, step, none}                   |
| `momentum`     | Float       | [0.0, 0.99] (used only when sgd)       |
| `epochs`       | Int         | [5, 30]                                |

The MLP is a 2-hidden-layer fully-connected network trained on standardized features with cross-entropy (classification only). Loss is `-validation_accuracy` so HPO minimizers see "lower is better."

**HPO objective for all sklearn tasks**: cross-validated score. We use `sklearn.model_selection.cross_val_score` with `cv=3` (3-fold stratified for classification, plain k-fold for regression). The HPO loss is $-\overline{\text{score}}$ for classification (`accuracy`) or $-\overline{\text{score}}$ for regression (`neg_mean_squared_error`, i.e. MSE).

### 2.3 Multi-fidelity adapters for sklearn / MLP

For HDGPSO-MF, the fidelity parameter modulates each task type:

| Model class | What `fidelity < 1` reduces |
|-------------|-----------------------------|
| RandomForest, GradientBoosting | `n_estimators` proportionally; CV folds reduced (3 → 2 → 1) at thresholds 0.6 and 0.5 |
| MLP         | Training set size and `epochs` proportionally |
| PINN        | `n_collocation` and `epochs` proportionally |

For `fidelity < 0.5` on sklearn tasks, we switch from CV to a single 75/25 train/test split for further speedup.

---

## 3. Hardware and Software

### 3.1 Hardware

| Component | Spec |
|-----------|------|
| GPU       | NVIDIA RTX 3500 Ada Generation Laptop, **12.9 GB VRAM** |
| CUDA / cuDNN | 12.8 / 91002 |
| CPU       | Intel/AMD x86\_64 (laptop class) |
| RAM       | 32 GB |
| Storage   | SSD (irrelevant — benchmark is compute-bound) |

GPU is used for **MLP** and **PINN** tasks (where PyTorch handles the heavy lifting). All sklearn cells (RandomForest, GradientBoosting) run on CPU.

### 3.2 Software stack

| Package            | Version    | Role |
|--------------------|------------|------|
| Python             | 3.13.9     | runtime |
| PyTorch            | 2.10.0+cu128 | MLP and PINN training, autograd |
| scikit-learn       | 1.6.1      | tree-ensemble models, CV utilities |
| NumPy              | 2.4.2      | array math |
| Pandas             | 2.2.3      | dataframe storage of trial history |
| SciPy              | 1.15.3     | Friedman test, Wilcoxon test |
| Matplotlib         | 3.10.0     | paper figures (bar / line / heatmap) |
| scikit-optimize    | 0.10.2     | Bayes baseline tuner |
| Optuna             | 4.8.0      | Optuna-TPE baseline tuner |
| pyswarms           | 1.3        | PSO baseline tuner |
| xgboost            | 3.2.0      | optional, not used in headline benchmark |

The `hdgpso` package itself has only **NumPy, Pandas, SciPy, scikit-learn** as required dependencies. Everything else is optional (used only by specific baselines).

### 3.3 Reproducibility

- All cells use `np.random.default_rng(seed)`, `torch.manual_seed(seed)`, and per-cell deterministic ordering.
- The benchmark seeds are `[0, 1, 2]` for the v5 main run.
- The total compute cost of the headline benchmark is approximately **18.5 wall-clock hours** on the hardware above (Section 3.1).
- The result CSVs (`results_claim_check_v5/summary.csv`, `results_budget_sweep/summary_budget_sweep.csv`) are tracked and uploaded with the paper materials.
- Each row of these CSVs contains the cell coordinates `(dataset, model, tuner, seed)` and the full trial history is in `history.csv`, sufficient to re-derive every statistical claim in the paper.

### 3.4 Why a laptop GPU?

We deliberately benchmark on a **laptop-class GPU** (RTX 3500 Ada Mobile, 12.9 GB) because most practical HPO workloads run on this hardware — not on cloud A100/H100 clusters. The benchmark conclusions (HDGPSO is best at budget=60; multi-fidelity helps when each evaluation is expensive) are most relevant to the budget-constrained settings researchers actually face.

---

## 4. Statistical Methodology in Detail — Demšar (2006)

This section unpacks the statistical machinery referenced in §V.D of the main paper. We follow Demšar (2006) — the canonical protocol for comparing $K$ algorithms across $N$ datasets.

### 4.1 Why rank-based statistics?

When comparing tuners across heterogeneous tasks (`accuracy` for classification, `MSE` for regression, autograd-residual MSE for PINN), the *absolute* loss values are on incomparable scales. Averaging them across tasks is meaningless ("on average HDGPSO scores 0.42") because 0.42 means very different things on `breast_cancer` vs `pinn_heat`.

**Ranks solve this.** Within each block (defined below), rank the tuners 1 (best) through K (worst). Ranks are dimensionless and comparable across tasks. A tuner with rank 2 on cell A and rank 3 on cell B is *consistently good*; an absolute-loss comparison would have hidden that.

### 4.2 The block structure: what is a "cell"?

A **cell** (= "block" in the statistics literature) is a unique combination of `(dataset, model, seed)`. For our benchmark at one budget:

- 9 compatible `(dataset, model)` pairs × 3 seeds = **27 blocks**
- Each block contains K = 8 tuner observations (one `best_loss` per tuner)
- Total cells = 27 × 8 = **216**

The blocks are the *unit of replication* for the rank-based tests. The 27 figure $(N = 27)$ enters the formulas below.

### 4.3 Friedman omnibus test

The Friedman test (Friedman, 1937) is a non-parametric, rank-based analog of repeated-measures ANOVA. It asks the **single global question**:

> *Are all K tuners equivalent in expected rank across the N blocks?*

**Procedure:**

1. For each block (one row of `summary.csv` after pivoting), rank the K tuners by `best_loss` from 1 (best) to K (worst). Tied values share their average rank.
2. Compute each tuner's **mean rank** $R_j$ across all N blocks.
3. Under the null hypothesis $H_0$: all tuners equivalent, the expected mean rank of every tuner is $(K + 1)/2$.
4. The Friedman statistic is

$$
\chi^2_F \;=\; \frac{12 N}{K(K+1)} \left[\sum_{j=1}^{K} R_j^2 \;-\; \frac{K(K+1)^2}{4}\right]
$$

5. Under $H_0$, $\chi^2_F$ is approximately $\chi^2$-distributed with $K - 1$ degrees of freedom.

**Our numbers (main run, $b = 60$):** $\chi^2_F = 79.66$ on 7 d.f., giving $p = 1.6 \times 10^{-14}$. We reject the global null overwhelmingly.

**Why "omnibus"?** The test signals that *some* tuners differ from others somewhere among the 28 possible pairs of 8 tuners — but **not which ones**. The Nemenyi post-hoc fills that role.

### 4.4 Nemenyi post-hoc test

Once Friedman rejects $H_0$, the **Nemenyi test** (Nemenyi, 1963 [Ph.D. thesis, Princeton]) tells us which pairs of tuners are significantly different.

**The Critical Difference (CD) threshold:**

$$
\boxed{\;\mathrm{CD}(K, N, \alpha) \;=\; q_\alpha \cdot \sqrt{\frac{K(K+1)}{6 N}}\;}
$$

where $q_\alpha$ is the studentized-range statistic at significance level $\alpha$. Two tuners with mean-rank difference $\;|R_A - R_B| > \mathrm{CD}\;$ are **significantly different**; otherwise they are statistically *equivalent* at that $\alpha$.

| $K$ | $q_{0.05}$ | $q_{0.10}$ |
|----:|----------:|----------:|
| 2   | 1.960     | 1.645     |
| 4   | 2.569     | 2.291     |
| 6   | 2.850     | 2.589     |
| **8**   | **3.031** | **2.780** |
| 10  | 3.164     | 2.920     |

For our headline ($K = 8$, $N = 27$, $\alpha = 0.05$):

$$
\mathrm{CD} \;=\; 3.031 \times \sqrt{\frac{8 \times 9}{6 \times 27}} \;\approx\; 2.02
$$

**Reading our CD diagram (paper Fig. 3 caption):** HDGPSO has mean rank 2.91. Any tuner with mean rank $> 2.91 + 2.02 = 4.93$ is Nemenyi-significantly worse:

| Tuner | Mean rank | Gap to HDGPSO | Nemenyi-significant at $\alpha = 0.05$? |
|---|---:|---:|:---|
| HDGPSO        | 2.91 | —    | (reference) |
| Bayes         | 3.20 | 0.30 | No (statistically tied) |
| Optuna-TPE    | 3.28 | 0.37 | No (statistically tied) |
| HDGPSO-MF     | 4.11 | 1.20 | No (within CD) |
| PSO           | 4.85 | 1.94 | No (within CD, just below threshold) |
| **DE**        | **4.96** | **2.05** | **Yes** |
| **RandomSearch** | **5.33** | **2.42** | **Yes** |
| **GridSearch** | **7.35** | **4.44** | **Yes** |

So at the strict $\alpha = 0.05$ Nemenyi threshold, HDGPSO is significantly better than **GridSearch, RandomSearch, and DE** (3 baselines). PSO is right on the edge (gap 1.94 vs CD 2.02); we report it as "tied" to be conservative.

**Why Nemenyi instead of pairwise t-tests?** Nemenyi controls the family-wise error rate across all $K(K-1)/2 = 28$ pairwise comparisons without the over-correction of Bonferroni-on-t-tests. It is also non-parametric (no normality assumption), matching the rank-based block structure.

### 4.5 Wilcoxon signed-rank as backup

We also report the **Wilcoxon signed-rank** test (Wilcoxon, 1945) for each (HDGPSO, baseline) pair as a sanity check. Wilcoxon operates on paired *loss differences* per block:

$$
W = \sum_{i=1}^{N} \mathrm{sgn}(d_i) \cdot \mathrm{rank}(|d_i|), \quad d_i = \mathcal{L}_{\text{baseline}, i} - \mathcal{L}_{\text{HDGPSO}, i}
$$

A small $p$-value indicates the median paired difference is non-zero. For HDGPSO vs Bayes we got $p = 0.90$ (no evidence of difference); vs Optuna-TPE $p = 0.29$ (similarly no evidence); vs DE $p = 6.7 \times 10^{-3}$ (clear difference). Wilcoxon agrees with Nemenyi on which pairs are "tied vs different" but uses a different statistical machinery, providing robustness.

### 4.6 Cliff's $\delta$ effect size

Statistical significance answers *"is the difference reproducible?"* It does **not** answer *"how big is the difference?"* For that, we report **Cliff's $\delta$** (Cliff, 1993), a non-parametric effect size in $[-1, +1]$:

$$
\delta(A, B) \;=\; P(L_A < L_B) - P(L_A > L_B)
$$

i.e., for each pair $(L_{A,i}, L_{B,j})$ across all $i, j$ block-tuples, count how often $A$ beats $B$ minus how often $B$ beats $A$, normalized.

**Conventional magnitude bins** (Romano, Kromrey, et al., 2006):

| $\|\delta\|$ | Interpretation |
|-----:|:----|
| < 0.147 | Negligible |
| < 0.330 | Small |
| < 0.474 | Medium |
| ≥ 0.474 | Large |

**Our paper:** vs the underperforming baselines, the largest effect is HDGPSO vs GridSearch ($\delta = +0.41$, **medium**); vs Bayes/Optuna it's $\delta = +0.02$ and $+0.05$ respectively (**negligible**). This is consistent with HDGPSO being a *clear improvement over naive search* but only a *trivial improvement over Bayes/Optuna* — exactly what the rank-tie story conveys.

### 4.7 Bootstrap rank confidence intervals

We report 95% bootstrap CIs (Efron, 1979) on per-tuner mean rank to show the rank ordering is stable across re-samplings of the cell pool, not an artifact of which specific 27 blocks we picked.

**Procedure:**

1. For each bootstrap iteration $b = 1, \dots, B$ (we use $B = 2000$): sample $N = 27$ blocks with replacement from the original 27 blocks.
2. Recompute the mean rank for each tuner on this resampled block set.
3. The 95% CI for tuner $j$ is the 2.5%–97.5% percentile of $\{R_j^{(b)}\}_{b=1}^{B}$.

The width of the CI indicates how sensitive the ordering is to the specific tasks chosen. HDGPSO's 95% CI was $[2.31, 3.50]$, narrowly overlapping with Bayes' $[2.61, 3.78]$ and Optuna-TPE's $[2.69, 3.83]$ — quantifying the "tie" qualitatively.

### 4.8 Why this protocol versus simpler alternatives?

| Alternative | Why we don't use it |
|--|--|
| Just average raw losses across datasets | Losses on incommensurable scales (accuracy vs MSE vs PDE residual MSE). |
| Pairwise paired $t$-tests | Assume normality; rank differences are not normal. |
| Pairwise $t$-tests + Bonferroni | Over-corrects with many pairs; loses power. |
| Friedman alone | Doesn't reveal *which* tuners differ. |
| Nemenyi alone, no Friedman | Inflates family-wise error if applied without rejection of the global null. |

The Demšar (2006) protocol — **Friedman omnibus first, Nemenyi post-hoc second, supplemented by Wilcoxon, Cliff's $\delta$, and bootstrap CIs** — is the *de facto* standard for ML algorithm-comparison papers. Reviewers expect to see it; deviation invites pushback.

### 4.9 Threats to statistical validity

- **Sample size.** $N = 27$ per budget is on the lower end. A rank gap of $\sim 0.3$ between HDGPSO and Bayes/Optuna cannot be resolved at $\alpha = 0.05$ with this $N$; the failed Claim 4 in the paper is a Type II error (insufficient power), not evidence of equivalence.
- **Multiple budget tests.** The budget sweep evaluates 4 budgets independently. We do not apply a family-wise correction across budgets because each budget is reported as a separate result, not as part of a joint hypothesis.
- **PINN as single-problem class.** Conclusions about PINN behavior rest on one PDE; do not generalize from the heat equation to all physics-informed networks without further benchmarking.

### 4.10 Code and reproducibility

All five tests are implemented in [`src/hdgpso/stats.py`](../src/hdgpso/stats.py):

```python
from hdgpso.stats import (
    friedman_test, nemenyi_matrix, critical_difference,
    hdgpso_vs_baselines_table, bootstrap_rank_ci, cliffs_delta,
)

summary = pd.read_csv("results_claim_check_v5/summary.csv")
print(friedman_test(summary))
print(bootstrap_rank_ci(summary, n_boot=2000))
print(hdgpso_vs_baselines_table(summary, target="HDGPSO"))
```

The full Demšar battery runs in $< 3$ seconds on the saved CSVs.

---

## 5. References (PINN-specific)

- Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.* Journal of Computational Physics, 378, 686–707. — the foundational PINN paper.
- Karniadakis, G. E., et al. (2021). *Physics-informed machine learning.* Nature Reviews Physics, 3, 422–440. — modern survey.
- Wang, S., Sankaran, S., & Perdikaris, P. (2022). *Respecting causality is all you need for training physics-informed neural networks.* arXiv:2203.07404. — discussion of loss-weighting strategies (which our HPO directly optimizes via `physics_weight` and `bc_weight`).

The 1D heat equation IBVP we use is a textbook example covered in any PDE course; see e.g. Strauss, *Partial Differential Equations: An Introduction* (2008), Chapter 2.
